# ATC Recorder — 日本ATC音声まとめ

このノートブックは LiveATC.net のストリームを録音し、GitHub リポジトリに自動アップロードします。

## 手順
1. **セル1**: GitHub トークンとリポジトリを設定
2. **セル2**: 必要なパッケージをインストール
3. **セル3**: 録音設定を入力
4. **セル4**: 録音を実行
5. **セル5**: GitHub にアップロード


In [ ]:
# ============================================================
# セル1: 設定 — GitHub トークンとリポジトリ名を入力
# ============================================================

from google.colab import userdata

# Colab の Secrets に GITHUB_TOKEN を登録しておくと安全です
# (左サイドバーの鍵アイコン → 「GITHUB_TOKEN」という名前で追加)
# または下の行を直接書き換えても OK（共有には注意）

try:
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    GITHUB_TOKEN = ''  # ← ここに直接入力することもできます

GITHUB_REPO  = 'yasuminu/flight-sounds'   # オーナー/リポジトリ名
GITHUB_BRANCH = 'main'

print(f'リポジトリ: {GITHUB_REPO}')
print(f'トークン設定済み: {bool(GITHUB_TOKEN)}')


In [ ]:
# ============================================================
# セル2: パッケージインストール
# ============================================================

import subprocess, sys

# ffmpeg (録音用)
subprocess.run(['apt-get', 'install', '-y', '-q', 'ffmpeg'], check=True)

# Python ライブラリ
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'requests', 'pydub'], check=True)

print('インストール完了')


In [ ]:
# ============================================================
# セル3: 録音設定
# ============================================================

# --- 空港・フィード情報 ---
AIRPORT_ICAO = 'RJAA'          # 空港 ICAO コード
AIRPORT_IATA = 'NRT'          # 空港 IATA コード
FEED_NAME    = 'NRT アプローチ/デパーチャー'
DESCRIPTION  = '成田アプローチ 着陸シーケンス'  # 録音の説明

# --- 録音するストリーム URL ---
# LiveATC.net のストリーム URL を入力してください
# 例: https://www.liveatc.net/search/?icao=RJAA から確認できます
# ストリーム URL は .pls / .m3u ファイルか直接 URL を指定
STREAM_URL   = 'https://s1-fmt2.liveatc.net/rjaa'   # ← 変更してください

# --- 録音時間 ---
DURATION_MINUTES = 5   # 録音する分数

print(f'空港: {AIRPORT_IATA} ({AIRPORT_ICAO})')
print(f'フィード: {FEED_NAME}')
print(f'ストリーム: {STREAM_URL}')
print(f'録音時間: {DURATION_MINUTES} 分')


In [ ]:
# ============================================================
# セル4: 録音実行
# ============================================================

import subprocess
from datetime import datetime, timezone, timedelta
import os

JST = timezone(timedelta(hours=9))
now = datetime.now(JST)
timestamp = now.strftime('%Y%m%d_%H%M')
date_str  = now.strftime('%Y-%m-%d')
time_str  = now.strftime('%H:%M')

filename = f'{AIRPORT_ICAO}_{timestamp}.mp3'
local_path = f'/tmp/{filename}'

print(f'録音開始: {filename}')
print(f'時刻 (JST): {date_str} {time_str}')
print(f'録音中... ({DURATION_MINUTES}分)')

result = subprocess.run(
    [
        'ffmpeg', '-y',
        '-i', STREAM_URL,
        '-t', str(DURATION_MINUTES * 60),
        '-acodec', 'libmp3lame',
        '-ab', '64k',       # 64kbps — 音声品質と容量のバランス
        '-ar', '22050',
        local_path
    ],
    capture_output=True, text=True
)

if result.returncode == 0 and os.path.exists(local_path):
    size_kb = os.path.getsize(local_path) // 1024
    print(f'録音完了: {filename} ({size_kb} KB)')
else:
    print('録音エラー:')
    print(result.stderr[-2000:] if result.stderr else '不明なエラー')
    raise RuntimeError('録音に失敗しました。ストリーム URL を確認してください。')


In [ ]:
# ============================================================
# セル5: GitHub へアップロード
# ============================================================

import requests
import base64
import json
import os

HEADERS = {
    'Authorization': f'token {GITHUB_TOKEN}',
    'Accept': 'application/vnd.github+json',
    'X-GitHub-Api-Version': '2022-11-28',
}
BASE_URL = f'https://api.github.com/repos/{GITHUB_REPO}'

def github_get_file_sha(path):
    """既存ファイルの SHA を取得（更新時に必要）"""
    r = requests.get(
        f'{BASE_URL}/contents/{path}',
        headers=HEADERS,
        params={'ref': GITHUB_BRANCH}
    )
    if r.status_code == 200:
        return r.json().get('sha')
    return None


def github_upload_file(path, content_bytes, commit_message):
    """ファイルをアップロード（新規 or 更新）"""
    sha = github_get_file_sha(path)
    payload = {
        'message': commit_message,
        'content': base64.b64encode(content_bytes).decode('utf-8'),
        'branch': GITHUB_BRANCH,
    }
    if sha:
        payload['sha'] = sha

    r = requests.put(
        f'{BASE_URL}/contents/{path}',
        headers=HEADERS,
        json=payload
    )
    r.raise_for_status()
    return r.json()


# --- 1. 音声ファイルをアップロード ---
audio_github_path = f'audio/{filename}'
print(f'音声ファイルをアップロード中: {audio_github_path}')

with open(local_path, 'rb') as f:
    audio_bytes = f.read()

github_upload_file(
    audio_github_path,
    audio_bytes,
    f'録音追加: {AIRPORT_ICAO} {date_str} {time_str}'
)
print('音声ファイルのアップロード完了')


# --- 2. recordings.json を取得して更新 ---
print('recordings.json を更新中...')

r = requests.get(
    f'{BASE_URL}/contents/data/recordings.json',
    headers=HEADERS,
    params={'ref': GITHUB_BRANCH}
)
r.raise_for_status()
json_data = json.loads(base64.b64decode(r.json()['content']).decode('utf-8'))

# 新しいエントリを先頭に追加
new_entry = {
    'id': f'rec_{timestamp}_{AIRPORT_ICAO}',
    'airport': AIRPORT_ICAO,
    'iata': AIRPORT_IATA,
    'feed': FEED_NAME,
    'date': date_str,
    'time': time_str,
    'duration': DURATION_MINUTES * 60,
    'filename': filename,
    'description': DESCRIPTION,
    'size': os.path.getsize(local_path),
}

json_data['recordings'].insert(0, new_entry)
updated_json = json.dumps(json_data, ensure_ascii=False, indent=2)

github_upload_file(
    'data/recordings.json',
    updated_json.encode('utf-8'),
    f'録音メタデータ更新: {AIRPORT_ICAO} {date_str} {time_str}'
)

print('recordings.json の更新完了')
print()
print('=== 完了 ===')
print(f'録音ファイル: {filename}')
print(f'GitHub: https://github.com/{GITHUB_REPO}/blob/{GITHUB_BRANCH}/{audio_github_path}')
print('Cloudflare Pages が自動デプロイされると、サイトに反映されます。')


---

## 手動でファイルをアップロードする場合

録音済みの MP3 ファイルがある場合は、下のセルを使ってください。


In [ ]:
# ============================================================
# 手動アップロード — すでに手元に MP3 ファイルがある場合
# ============================================================

from google.colab import files
import requests, base64, json, os
from datetime import datetime, timezone, timedelta

# ファイルを選択してアップロード
print('アップロードするファイルを選択してください')
uploaded = files.upload()

JST = timezone(timedelta(hours=9))
now = datetime.now(JST)
date_str = now.strftime('%Y-%m-%d')
time_str = now.strftime('%H:%M')

HEADERS = {
    'Authorization': f'token {GITHUB_TOKEN}',
    'Accept': 'application/vnd.github+json',
    'X-GitHub-Api-Version': '2022-11-28',
}
BASE_URL = f'https://api.github.com/repos/{GITHUB_REPO}'

# --- 各ファイルを処理 ---
for local_filename, content_bytes in uploaded.items():
    # 録音情報（変更してください）
    manual_airport_icao = input(f'{local_filename} の空港 ICAO コード (例: RJAA): ').strip() or 'RJAA'
    manual_airport_iata = input(f'{local_filename} の空港 IATA コード (例: NRT): ').strip() or 'NRT'
    manual_feed = input(f'フィード名 (例: NRT アプローチ): ').strip() or 'アプローチ'
    manual_description = input(f'説明: ').strip() or local_filename

    github_path = f'audio/{local_filename}'
    sha = None
    r = requests.get(f'{BASE_URL}/contents/{github_path}', headers=HEADERS, params={'ref': GITHUB_BRANCH})
    if r.status_code == 200:
        sha = r.json().get('sha')

    payload = {
        'message': f'手動アップロード: {local_filename}',
        'content': base64.b64encode(content_bytes).decode('utf-8'),
        'branch': GITHUB_BRANCH,
    }
    if sha:
        payload['sha'] = sha

    r = requests.put(f'{BASE_URL}/contents/{github_path}', headers=HEADERS, json=payload)
    r.raise_for_status()
    print(f'アップロード完了: {github_path}')

    # recordings.json を更新
    r2 = requests.get(f'{BASE_URL}/contents/data/recordings.json', headers=HEADERS, params={'ref': GITHUB_BRANCH})
    r2.raise_for_status()
    json_data = json.loads(base64.b64decode(r2.json()['content']).decode('utf-8'))
    rec_id = f'rec_{now.strftime("%Y%m%d_%H%M")}_{manual_airport_icao}'
    json_data['recordings'].insert(0, {
        'id': rec_id,
        'airport': manual_airport_icao,
        'iata': manual_airport_iata,
        'feed': manual_feed,
        'date': date_str,
        'time': time_str,
        'duration': 0,
        'filename': local_filename,
        'description': manual_description,
        'size': len(content_bytes),
    })

    updated_json = json.dumps(json_data, ensure_ascii=False, indent=2).encode('utf-8')
    sha2 = r2.json().get('sha')
    payload2 = {
        'message': f'手動アップロード metadata: {local_filename}',
        'content': base64.b64encode(updated_json).decode('utf-8'),
        'branch': GITHUB_BRANCH,
        'sha': sha2,
    }
    r3 = requests.put(f'{BASE_URL}/contents/data/recordings.json', headers=HEADERS, json=payload2)
    r3.raise_for_status()
    print('recordings.json を更新しました')

print('全ファイルのアップロードが完了しました。')


---

## ストリーム URL の探し方

1. [LiveATC.net](https://www.liveatc.net/) にアクセス
2. 聴きたい空港を検索（例: RJAA, RJTT）
3. フィードを選択してから「Listen」ボタンの URL を調べる
4. `.pls` ファイルをダウンロードしてテキストで開くと URL が確認できる

例:
```
# 成田 アプローチ
STREAM_URL = 'https://s1-fmt2.liveatc.net/rjaa'

# 羽田
STREAM_URL = 'https://s1-fmt2.liveatc.net/rjtt'
```

## GitHub Personal Access Token の作成方法

1. GitHub → Settings → Developer settings → Personal access tokens → Tokens (classic)
2. 「Generate new token (classic)」をクリック
3. スコープ: `repo` にチェック
4. 生成されたトークンをコピー
5. Google Colab の左サイドバー 鍵アイコン → `GITHUB_TOKEN` という名前で登録
